# 可选实验 - 带正则化的代价与梯度

## 目标
在本实验中，你将：
- 用正则化项扩展前面的线性与逻辑代价函数。
- 在加入正则化项的情况下重新运行前面的过拟合示例。


In [ ]:
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt
from plt_overfit import overfit_example, output
from lab_utils_common import sigmoid
np.set_printoptions(precision=8)

# 添加正则化
<img align="Left" src="./图片/C1_W3_LinearGradientRegularized.png"  style=" width:400px; padding: 10px; " >

> **图中文字（中文）**：正则化线性回归 min (1/2m)Σ(…) + (λ/2m)Σ w_j² 梯度下降 Repeat { … }
<img align="Center" src="./图片/C1_W3_LogisticGradientRegularized.png"  style=" width:400px; padding: 10px; " >

> **图中文字（中文）**：正则化逻辑回归 梯度下降 Repeat { … }（λ > 0 时，w_j 的更新项为 w_j − α·(1/m)Σ(…) − α·(λ/m)·w_j）

上面的幻灯片展示了线性回归和逻辑回归的代价函数与梯度函数。注意：
- 代价
    - 线性回归与逻辑回归的代价函数差别很大，但在方程中加入正则化的方式是相同的。
- 梯度
    - 线性回归与逻辑回归的梯度函数非常相似，区别只在于 $f_{wb}$ 的实现。

## 带正则化的代价函数
### 带正则化线性回归的代价函数

带正则化的线性回归代价函数方程为：
$$J(\\mathbf{w},b) = \\frac{1}{2m} \\sum\\limits_{i = 0}^{m-1} (f_{\\mathbf{w},b}(\\mathbf{x}^{(i)}) - y^{(i)})^2  + \\frac{\\lambda}{2m}  \\sum_{j=0}^{n-1} w_j^2 \\tag{1}$$ 
其中：
$$ f_{\\mathbf{w},b}(\\mathbf{x}^{(i)}) = \\mathbf{w} \\cdot \\mathbf{x}^{(i)} + b  \\tag{2} $$ 


把它与不带正则化的代价函数（你在之前的实验中实现过）比较，后者形式为：

$$J(\\mathbf{w},b) = \\frac{1}{2m} \\sum\\limits_{i = 0}^{m-1} (f_{\\mathbf{w},b}(\\mathbf{x}^{(i)}) - y^{(i)})^2 $$ 

区别在于正则化项  <span style="color:blue">
    $\\frac{\\lambda}{2m}  \\sum_{j=0}^{n-1} w_j^2$ </span> 
    
加入这一项会促使梯度下降最小化参数的大小。注意，在这个例子中参数 $b$ 不做正则化。这是标准做法。

下面是公式 (1) 和 (2) 的一个实现。注意它使用了*本课程的标准写法*，即对所有 `m` 个样本使用 `for 循环`。

In [ ]:
def compute_cost_linear_reg(X, y, w, b, lambda_ = 1):
    """计算所有样本上的代价
    参数：
      X (ndarray (m,n): 数据，m 个样本，n 个特征
      y (ndarray (m,)): 目标值
      w (ndarray (n,)): 模型参数
      b (scalar)      : 模型参数
      lambda_ (scalar): 控制正则化强度
    返回：
      total_cost (scalar):  代价"""

    m  = X.shape[0]
    n  = len(w)
    cost = 0.
    for i in range(m):
        f_wb_i = np.dot(X[i], w) + b                                   #(n,)(n,)=标量，见 np.dot
        cost = cost + (f_wb_i - y[i])**2                               #标量
    cost = cost / (2 * m)                                              #标量
 
    reg_cost = 0
    for j in range(n):
        reg_cost += (w[j]**2)                                          #标量
    reg_cost = (lambda_/(2*m)) * reg_cost                              #标量
    
    total_cost = cost + reg_cost                                       #标量
    return total_cost                                                  #标量

运行下面的单元格看看实际效果。

In [ ]:
np.random.seed(1)
X_tmp = np.random.rand(5,6)
y_tmp = np.array([0,1,0,1,0])
w_tmp = np.random.rand(X_tmp.shape[1]).reshape(-1,)-0.5
b_tmp = 0.5
lambda_tmp = 0.7
cost_tmp = compute_cost_linear_reg(X_tmp, y_tmp, w_tmp, b_tmp, lambda_tmp)

print("Regularized cost:", cost_tmp)

**预期输出**：
<table>
  <tr>
    <td> <b>正则化代价： </b> 0.07917239320214275 </td>
  </tr>
</table>

### 带正则化逻辑回归的代价函数
对于带正则化的**逻辑**回归，代价函数的形式为
$$J(\\mathbf{w},b) = \\frac{1}{m}  \\sum_{i=0}^{m-1} \\left[ -y^{(i)} \\log\\left(f_{\\mathbf{w},b}\\left( \\mathbf{x}^{(i)} \\right) \\right) - \\left( 1 - y^{(i)}\\right) \\log \\left( 1 - f_{\\mathbf{w},b}\\left( \\mathbf{x}^{(i)} \\right) \\right) \\right] + \\frac{\\lambda}{2m}  \\sum_{j=0}^{n-1} w_j^2 \\tag{3}$$
其中：
$$ f_{\\mathbf{w},b}(\\mathbf{x}^{(i)}) = sigmoid(\\mathbf{w} \\cdot \\mathbf{x}^{(i)} + b)  \\tag{4} $$ 

把它与不带正则化的代价函数（你在之前的实验中实现过）比较：

$$ J(\\mathbf{w},b) = \\frac{1}{m}\\sum_{i=0}^{m-1} \\left[ (-y^{(i)} \\log\\left(f_{\\mathbf{w},b}\\left( \\mathbf{x}^{(i)} \\right) \\right) - \\left( 1 - y^{(i)}\\right) \\log \\left( 1 - f_{\\mathbf{w},b}\\left( \\mathbf{x}^{(i)} \\right) \\right)\\right] $$

与上面线性回归的情形一样，区别在于正则化项，即    <span style="color:blue">
    $\\frac{\\lambda}{2m}  \\sum_{j=0}^{n-1} w_j^2$ </span> 

加入这一项会促使梯度下降最小化参数的大小。注意，在这个例子中参数 $b$ 不做正则化。这是标准做法。

In [ ]:
def compute_cost_logistic_reg(X, y, w, b, lambda_ = 1):
    """计算所有样本上的代价
    参数：
    参数：
      X (ndarray (m,n): 数据，m 个样本，n 个特征
      y (ndarray (m,)): 目标值
      w (ndarray (n,)): 模型参数
      b (scalar)      : 模型参数
      lambda_ (scalar): 控制正则化强度
    返回：
      total_cost (scalar):  代价"""

    m,n  = X.shape
    cost = 0.
    for i in range(m):
        z_i = np.dot(X[i], w) + b                                      #(n,)(n,)=标量，见 np.dot
        f_wb_i = sigmoid(z_i)                                          #标量
        cost +=  -y[i]*np.log(f_wb_i) - (1-y[i])*np.log(1-f_wb_i)      #标量
             
    cost = cost/m                                                      #标量

    reg_cost = 0
    for j in range(n):
        reg_cost += (w[j]**2)                                          #标量
    reg_cost = (lambda_/(2*m)) * reg_cost                              #标量
    
    total_cost = cost + reg_cost                                       #标量
    return total_cost                                                  #标量

运行下面的单元格看看实际效果。

In [ ]:
np.random.seed(1)
X_tmp = np.random.rand(5,6)
y_tmp = np.array([0,1,0,1,0])
w_tmp = np.random.rand(X_tmp.shape[1]).reshape(-1,)-0.5
b_tmp = 0.5
lambda_tmp = 0.7
cost_tmp = compute_cost_logistic_reg(X_tmp, y_tmp, w_tmp, b_tmp, lambda_tmp)

print("Regularized cost:", cost_tmp)

**预期输出**：
<table>
  <tr>
    <td> <b>正则化代价： </b> 0.6850849138741673 </td>
  </tr>
</table>

## 带正则化的梯度下降
运行梯度下降的基本算法不因正则化而改变，它是：
$$\\begin{align*}
&\\text{repeat until convergence:} \\; \\lbrace \\\\
&  \\; \\; \\;w_j = w_j -  \\alpha \\frac{\\partial J(\\mathbf{w},b)}{\\partial w_j} \\tag{1}  \\; & \\text{for j := 0..n-1} \\\\ 
&  \\; \\; \\;  \\; \\;b = b -  \\alpha \\frac{\\partial J(\\mathbf{w},b)}{\\partial b} \\\\
&\\rbrace
\\end{align*}$$
其中每次迭代对所有 $j$ 的 $w_j$ 同时更新。

正则化带来变化的是梯度的计算。

### 带正则化的梯度计算（线性/逻辑均适用）
线性回归与逻辑回归的梯度计算几乎完全相同，区别只在于 $f_{\\mathbf{w}b}$ 的计算。
$$\\begin{align*}
\\frac{\\partial J(\\mathbf{w},b)}{\\partial w_j}  &= \\frac{1}{m} \\sum\\limits_{i = 0}^{m-1} (f_{\\mathbf{w},b}(\\mathbf{x}^{(i)}) - y^{(i)})x_{j}^{(i)}  +  \\frac{\\lambda}{m} w_j \\tag{2} \\\\
\\frac{\\partial J(\\mathbf{w},b)}{\\partial b}  &= \\frac{1}{m} \\sum\\limits_{i = 0}^{m-1} (f_{\\mathbf{w},b}(\\mathbf{x}^{(i)}) - y^{(i)}) \\tag{3} 
\\end{align*}$$

* m 是数据集中的训练样本数
* $f_{\\mathbf{w},b}(x^{(i)})$ 是模型的预测，而 $y^{(i)}$ 是目标值

      
* 对于 <span style="color:blue"> **线性** </span> 回归模型
    $f_{\\mathbf{w},b}(x) = \\mathbf{w} \\cdot \\mathbf{x} + b$  
* 对于 <span style="color:blue"> **逻辑** </span> 回归模型
    $z = \\mathbf{w} \\cdot \\mathbf{x} + b$  
    $f_{\\mathbf{w},b}(x) = g(z)$  
    其中 $g(z)$ 是 sigmoid 函数：  
    $g(z) = \\frac{1}{1+e^{-z}}$  
    
添加正则化的项是 <span style="color:blue">$\\frac{\\lambda}{m} w_j $</span>。

### 带正则化线性回归的梯度函数

In [ ]:
def compute_gradient_linear_reg(X, y, w, b, lambda_): 
    """计算线性回归的梯度
    参数：
      X (ndarray (m,n): 数据，m 个样本，n 个特征
      y (ndarray (m,)): 目标值
      w (ndarray (n,)): 模型参数
      b (scalar)      : 模型参数
      lambda_ (scalar): 控制正则化强度
     
    返回：
      dj_dw (ndarray (n,)): 代价对参数 w 的梯度。
      dj_db (scalar):       代价对参数 b 的梯度。"""
    m,n = X.shape           #（样本数，特征数）
    dj_dw = np.zeros((n,))
    dj_db = 0.

    for i in range(m):                             
        err = (np.dot(X[i], w) + b) - y[i]                 
        for j in range(n):                         
            dj_dw[j] = dj_dw[j] + err * X[i, j]               
        dj_db = dj_db + err                        
    dj_dw = dj_dw / m                                
    dj_db = dj_db / m   
    
    for j in range(n):
        dj_dw[j] = dj_dw[j] + (lambda_/m) * w[j]

    return dj_db, dj_dw

运行下面的单元格看看实际效果。

In [ ]:
np.random.seed(1)
X_tmp = np.random.rand(5,3)
y_tmp = np.array([0,1,0,1,0])
w_tmp = np.random.rand(X_tmp.shape[1])
b_tmp = 0.5
lambda_tmp = 0.7
dj_db_tmp, dj_dw_tmp =  compute_gradient_linear_reg(X_tmp, y_tmp, w_tmp, b_tmp, lambda_tmp)

print(f"dj_db: {dj_db_tmp}", )
print(f"Regularized dj_dw:\n {dj_dw_tmp.tolist()}", )

**预期输出**

```
dj_db: 0.6648774569425726
Regularized dj_dw:
 [0.29653214748822276, 0.4911679625918033, 0.21645877535865857]
 ```


### 带正则化逻辑回归的梯度函数

In [ ]:
def compute_gradient_logistic_reg(X, y, w, b, lambda_): 
    """计算线性回归的梯度
 
    参数：
      X (ndarray (m,n): 数据，m 个样本，n 个特征
      y (ndarray (m,)): 目标值
      w (ndarray (n,)): 模型参数
      b (scalar)      : 模型参数
      lambda_ (scalar): 控制正则化强度
    返回
      dj_dw (ndarray 形状 (n,)): 代价对参数 w 的梯度。
      dj_db (scalar)            : 代价对参数 b 的梯度。"""
    m,n = X.shape
    dj_dw = np.zeros((n,))                            #(n,)
    dj_db = 0.0                                       #标量

    for i in range(m):
        f_wb_i = sigmoid(np.dot(X[i],w) + b)          #(n,)(n,)=标量
        err_i  = f_wb_i  - y[i]                       #标量
        for j in range(n):
            dj_dw[j] = dj_dw[j] + err_i * X[i,j]      #标量
        dj_db = dj_db + err_i
    dj_dw = dj_dw/m                                   #(n,)
    dj_db = dj_db/m                                   #标量

    for j in range(n):
        dj_dw[j] = dj_dw[j] + (lambda_/m) * w[j]

    return dj_db, dj_dw  


运行下面的单元格看看实际效果。

In [ ]:
np.random.seed(1)
X_tmp = np.random.rand(5,3)
y_tmp = np.array([0,1,0,1,0])
w_tmp = np.random.rand(X_tmp.shape[1])
b_tmp = 0.5
lambda_tmp = 0.7
dj_db_tmp, dj_dw_tmp =  compute_gradient_logistic_reg(X_tmp, y_tmp, w_tmp, b_tmp, lambda_tmp)

print(f"dj_db: {dj_db_tmp}", )
print(f"Regularized dj_dw:\n {dj_dw_tmp.tolist()}", )

**预期输出**

```
dj_db: 0.341798994972791
Regularized dj_dw:
 [0.17380012933994293, 0.32007507881566943, 0.10776313396851499]
 ```


## 重新运行过拟合示例

In [ ]:
plt.close("all")
display(output)
ofit = overfit_example(True)

在上图中，试着对前面的例子应用正则化。具体来说：
- 分类（逻辑回归）
    - 把次数设为 6，lambda 设为 0（无正则化），拟合数据
    - 现在把 lambda 设为 1（增大正则化），拟合数据，注意差别。
- 回归（线性回归）
    - 尝试同样的步骤。

## 恭喜！
你已经：
- 得到了线性回归和逻辑回归各自的代价与梯度程序示例（已加入正则化）
- 对正则化如何减轻过拟合建立了一些直觉